In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import math

print("GOLD LAYER: Feature Engineering")
print("="*60)

# Load from Silver
path = "abfss://8b73c65d-76d6-466c-96b4-ed517828198f@onelake.dfs.fabric.microsoft.com/6845437a-d5ca-4b69-a0ea-85531d87c90e/Tables/dbo/silver_crashes_cleaned"
df_silver = spark.read.format("delta").load(path)

print(f"Loaded from Silver: {df_silver.count():,} records")
print(f"Input columns: {len(df_silver.columns)}")

# Copy to gold
df_gold = df_silver

print("\nData loaded successfully")

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 10, Finished, Available, Finished, False)

GOLD LAYER: Feature Engineering
Loaded from Silver: 111,657 records
Input columns: 95

Data loaded successfully


In [9]:
print("\nCreating TEMPORAL features...")

# crashSHDescription is the State Highway flag
print("\nState Highway crashes:")
df_gold.groupBy("crashSHDescription").count().show()

# Parse "2014/2015" string format
df_gold = df_gold.withColumn(
    "financial_year_start",
    F.split(F.col("crashFinancialYear"), "/").getItem(0).cast("integer")
)

# NOTE: No crash hour in CAS public data — remove time_period feature
# Use is_state_highway and financial year instead

# Holiday type for richer temporal context
df_gold = df_gold.withColumn(
    "holiday_type",
    F.coalesce(F.col("holiday"), F.lit("None"))
)
# Holiday values: Easter | Christmas New Year | Labour Weekend | Queens Birthday | None


print("  Created temporal features:")
print("     - holiday_type")

# Show distribution
print("\n  Holiday distribution:")
df_gold.groupBy("holiday").count().orderBy(F.desc("count")).show()
print("\n  Yearly distribution:")
df_gold.groupBy("financial_year_start").count().orderBy(F.desc("count")).show()

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 11, Finished, Available, Finished, False)


Creating TEMPORAL features...

State Highway crashes:
+------------------+-----+
|crashSHDescription|count|
+------------------+-----+
|           Unknown|   10|
|                No|91160|
|               Yes|20487|
+------------------+-----+

  Created temporal features:
     - holiday_type

  Holiday distribution:
+------------------+------+
|           holiday| count|
+------------------+------+
|              NULL|106283|
|Christmas New Year|  2326|
|            Easter|  1157|
|   Queens Birthday|  1030|
|    Labour Weekend|   861|
+------------------+------+


  Yearly distribution:
+--------------------+-----+
|financial_year_start|count|
+--------------------+-----+
|                2016|12957|
|                2017|12455|
|                2015|11381|
|                2018|10982|
|                2020|10497|
|                2019| 9611|
|                2022| 9427|
|                2024| 8530|
|                2023| 8359|
|                2021| 8330|
|                2014| 5074

In [10]:
print("\nCreating SPATIAL features...")

# Auckland CBD coordinates
AUCKLAND_CBD_LAT = -36.8485
AUCKLAND_CBD_LON = 174.7633

# 1. Distance from CBD (Haversine approximation)
# Simplified: 1 degree latitude ≈ 111 km
df_gold = df_gold.withColumn(
    "distance_from_cbd_km",
    F.round(
        F.sqrt(
            F.pow((F.col("Y") - F.lit(AUCKLAND_CBD_LAT)) * 111, 2) +
            F.pow((F.col("X") - F.lit(AUCKLAND_CBD_LON)) * 111 * F.cos(F.radians(F.col("Y"))), 2)
        ),
        2
    )
)

# 2. Urban/Suburban/Rural classification (based on speed limit)
df_gold = df_gold.withColumn(
    "area_type",
    F.when(F.col("speedLimit").isNull(), "Unknown")
     .when(F.col("speedLimit") <= 50, "Urban")
     .when(F.col("speedLimit") <= 70, "Suburban")
     .otherwise("Rural")
)

# 3. CBD proximity flag (within 5km)
df_gold = df_gold.withColumn(
    "is_cbd_area",
    F.when(F.col("distance_from_cbd_km") <= 5, 1).otherwise(0)
)

print("  Created spatial features:")
print("     - distance_from_cbd_km")
print("     - area_type")
print("     - is_cbd_area")

# Show distribution
print("\n  Area type distribution:")
df_gold.groupBy("area_type").count().orderBy(F.desc("count")).show()

print("\n  Distance from CBD statistics:")
df_gold.select(
    F.min("distance_from_cbd_km").alias("min_km"),
    F.avg("distance_from_cbd_km").alias("avg_km"),
    F.max("distance_from_cbd_km").alias("max_km")
).show()

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 12, Finished, Available, Finished, False)


Creating SPATIAL features...
  Created spatial features:
     - distance_from_cbd_km
     - area_type
     - is_cbd_area

  Area type distribution:
+---------+-----+
|area_type|count|
+---------+-----+
|    Urban|78103|
|    Rural|25290|
| Suburban| 7815|
|  Unknown|  449|
+---------+-----+


  Distance from CBD statistics:
+------+------------------+------+
|min_km|            avg_km|max_km|
+------+------------------+------+
|  0.01|12.463485674879317| 45.78|
+------+------------------+------+



In [11]:
print("\nCreating INTERACTION features...")

# Values: Fine | Hail or Sleet | Heavy Rain | Light Rain | Mist or Fog | Snow | Null

df_gold = df_gold.withColumn(
    "wet_and_dark",
    F.when(
        F.col("weatherA").isin(["Light Rain", "Heavy Rain", "Mist or Fog"]) &
        (F.col("light") == "Dark"),
        1
    ).otherwise(0)
)

df_gold = df_gold.withColumn(
    "poor_visibility",
    F.when(
        F.col("weatherA").isin(["Mist or Fog", "Heavy Rain", "Snow", "Hail or Sleet"]) |
        (F.col("light") == "Dark") |
        (F.col("weatherB_clean") == "Frost") |
        (F.col("weatherB_clean") == "Strong Wind"),
        1
    ).otherwise(0)
)

# NEW — frost risk from weatherB (Frost / None / Null / Strong Wind)
df_gold = df_gold.withColumn(
    "frost_risk",
    F.when(F.col("weatherB_clean") == "Frost", 1).otherwise(0)
)

# CORRECTED — flatHill replaces roadCurvature
df_gold = df_gold.withColumn(
    "high_speed_hill",
    F.when(
        (F.coalesce(F.col("speedLimit"), F.lit(0)) >= 80) &
        (F.col("flatHill_clean") == "Hill Road"),
        1
    ).otherwise(0)
)

# NEW — advisory speed delta
df_gold = df_gold.withColumn(
    "advisory_speed_delta",
    F.when(
        F.col("advisorySpeed").isNotNull() & F.col("speedLimit").isNotNull(),
        F.col("speedLimit") - F.col("advisorySpeed")
    ).otherwise(0)
)

print("  Created interaction features:")
print("     - wet_and_dark")
print("     - poor_visibility")
print("     - frost_risk")
print("     - high_speed_hill")
print("     - advisory_speed_delta")


StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 13, Finished, Available, Finished, False)


Creating INTERACTION features...
  Created interaction features:
     - wet_and_dark
     - poor_visibility
     - frost_risk
     - high_speed_hill
     - advisory_speed_delta


In [12]:
print("\nCreating DERIVED features...")

# 1. Multi-vehicle flag
df_gold = df_gold.withColumn(
    "multi_vehicle",
    F.when(F.col("vehicle") > 1, 1).otherwise(0)
)

# trafficControl as intersection proxy (intersection col is always Blank)
# trafficControl values: Give way | Unknown | Nil | Pointsman | School Patrol | Stop | Traffic Signals
df_gold = df_gold.withColumn(
    "is_intersection",
    F.when(
        F.col("trafficControl").isNotNull() & 
        (~F.col("trafficControl").isin(["Nil", "Unknown"])),
        1
    ).otherwise(0)
)

# exact CAS trafficControl values
df_gold = df_gold.withColumn(
    "complex_traffic_control",
    F.when(
        F.col("trafficControl").isin(["Traffic Signals", "School Patrol", "Pointsman"]),
        1
    ).otherwise(0)
)

# road character flag (Nil/Broadway Ramp/Overpass/Rail xing/Tunnel/Underpass etc.)
df_gold = df_gold.withColumn(
    "is_special_road_feature",
    F.when(
        F.col("roadCharacter").isin(
            ["Overpass", "Underpass", "Tunnel", "Rail xing", "Broadway Ramp"]
        ), 1
    ).otherwise(0)
)

# road lane flags
df_gold = df_gold.withColumn(
    "is_off_road",
    F.when(F.col("roadLane_clean") == "Off road", 1).otherwise(0)
)
df_gold = df_gold.withColumn(
    "is_one_way",
    F.when(F.col("roadLane_clean") == "1 way", 1).otherwise(0)
)

print("  Created derived features:")
print("     - multi_vehicle")
print("     - is_intersection")
print("     - complex_traffic_control")
print("     - is_special_road_feature")
print("     - is_off_road")
print("     - is_one_way")

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 14, Finished, Available, Finished, False)


Creating DERIVED features...
  Created derived features:
     - multi_vehicle
     - is_intersection
     - complex_traffic_control
     - is_special_road_feature
     - is_off_road
     - is_one_way


In [13]:
print("\nCreating TARGET variables...")

# 1. Binary severity (for ML classification)
df_gold = df_gold.withColumn(
    "severe_crash",
    F.when(F.col("crashSeverity_clean").isin(["Fatal", "Serious"]), 1).otherwise(0)
)

# 2. Severity numeric (for ordered analysis)
severity_map = {"Non-Injury": 0, "Minor": 1, "Serious": 2, "Fatal": 3}

df_gold = df_gold.withColumn(
    "severity_numeric",
    F.when(F.col("crashSeverity_clean") == "Non-Injury", 0)
     .when(F.col("crashSeverity_clean") == "Minor", 1)
     .when(F.col("crashSeverity_clean") == "Serious", 2)
     .when(F.col("crashSeverity_clean") == "Fatal", 3)
     .otherwise(None)
)

# 3. Fatality flag
df_gold = df_gold.withColumn(
    "is_fatal",
    F.when(F.col("crashSeverity_clean") == "Fatal", 1).otherwise(0)
)

print("  Created target variables:")
print("     - severe_crash (binary: 0/1)")
print("     - severity_numeric (ordinal: 0-3)")
print("     - is_fatal (binary: 0/1)")

# Show target distribution
print("\n  Target variable distributions:")
print("\n  Severe crash:")
df_gold.groupBy("severe_crash").count().show()

print("  Severity numeric:")
df_gold.groupBy("severity_numeric", "crashSeverity_clean").count().orderBy("severity_numeric").show()

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 15, Finished, Available, Finished, False)


Creating TARGET variables...
  Created target variables:
     - severe_crash (binary: 0/1)
     - severity_numeric (ordinal: 0-3)
     - is_fatal (binary: 0/1)

  Target variable distributions:

  Severe crash:
+------------+------+
|severe_crash| count|
+------------+------+
|           1|  5363|
|           0|106294|
+------------+------+

  Severity numeric:
+----------------+-------------------+-----+
|severity_numeric|crashSeverity_clean|count|
+----------------+-------------------+-----+
|            NULL|            Unknown|80243|
|               1|              Minor|26051|
|               2|            Serious| 4986|
|               3|              Fatal|  377|
+----------------+-------------------+-----+



In [14]:
print("\nFeature engineering summary...")

# Count features
input_cols = len(df_silver.columns)
output_cols = len(df_gold.columns)
engineered_features = output_cols - input_cols

print("\nFEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"Input columns (Silver): {input_cols}")
print(f"Output columns (Gold): {output_cols}")
print(f"Engineered features: {engineered_features}")

# List new features
new_features = [col for col in df_gold.columns if col not in df_silver.columns]
print(f"\nNew features created ({len(new_features)}):")
for i, feat in enumerate(new_features, 1):
    print(f"  {i:2d}. {feat}")

# Final data quality check
print("\nFinal data quality:")
print(f"  Total records: {df_gold.count():,}")
print(f"  Null severe_crash: {df_gold.filter(F.col('severe_crash').isNull()).count()}")
print(f"  Null coordinates: {df_gold.filter(F.col('X').isNull() | F.col('Y').isNull()).count()}")

# Write to Gold table
print("\nWriting Gold table...")

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_crashes_features")

print("Gold table created: gold_crashes_features")

# Verify
verify_count = spark.table("gold_crashes_features").count()
print(f"Verified: {verify_count:,} records")

print("\nGOLD LAYER FEATURE ENGINEERING COMPLETE!")
print(f"Ready for analysis with {output_cols} total features")

StatementMeta(, 3760e3ce-5f8b-4532-86ab-02e5a419d54d, 16, Finished, Available, Finished, False)


Feature engineering summary...

FEATURE ENGINEERING SUMMARY
Input columns (Silver): 95
Output columns (Gold): 114
Engineered features: 19

New features created (19):
   1. financial_year_start
   2. holiday_type
   3. distance_from_cbd_km
   4. area_type
   5. is_cbd_area
   6. wet_and_dark
   7. poor_visibility
   8. frost_risk
   9. high_speed_hill
  10. advisory_speed_delta
  11. multi_vehicle
  12. is_intersection
  13. complex_traffic_control
  14. is_special_road_feature
  15. is_off_road
  16. is_one_way
  17. severe_crash
  18. severity_numeric
  19. is_fatal

Final data quality:
  Total records: 111,657
  Null severe_crash: 0
  Null coordinates: 0

Writing Gold table...
Gold table created: gold_crashes_features
Verified: 111,657 records

GOLD LAYER FEATURE ENGINEERING COMPLETE!
Ready for analysis with 114 total features
